# Observatorio CMF Chile: actualización anual de datos

Una vez al año, cuando el DEIS o FONASA publican un año nuevo:

1. **Aquí:** clasifica las bases DEIS y GRD en este computador (no salen de él), revisa la app en tu navegador y
   genera un zip con los casos clasificados (sin identificadores ni fechas) y las tablas publicadas.
2. **Panel de administración → Publicar → "Actualización anual":** sube el zip. Revisa la tabla de controles.
3. **Panel → "Desplegar y versionar":** "Solo redesplegar" y crea la versión (*minor*). Zenodo la agrega bajo el
   **mismo DOI de concepto**, así las citas no se dispersan.

Los **parámetros** (IPC, productividad, población INE, nacidos vivos, aranceles MAI, oferta) y los **precios GRD**
no necesitan este notebook: se suben en el panel (*Cargar datos* y *Diccionarios*) y se aplican con "Recalcular",
que corre en GitHub sobre los casos ya clasificados.

Requisitos: R ≥ 4.3 con los paquetes del observatorio (data.table, yaml, jsonlite, readxl, digest) y Python con pandas.
Los años ya procesados quedan en caché: la corrida siguiente solo procesa lo nuevo o lo que cambió.

In [1]:
from pathlib import Path
import datetime, json, shutil, subprocess, os

# --- Rutas en este computador (las mismas de tus notebooks) -----------------------------------------------
# El repositorio se detecta solo si abres este notebook desde su carpeta notebooks/; si no, se busca en Descargas.
_cands = [Path.cwd().parent, Path.cwd(), Path(r"C:/Users/gusta/Downloads/observatorio CMF/observatoriocmf"),
          Path(r"C:/Users/gusta/Downloads/observatoriocmf")]
REPO_DIR = next((c for c in _cands if (c / "pipeline" / "actualizar_local.R").exists()), _cands[2])

DEIS_DIR = Path(r"C:/Users/gusta/Downloads/Estudio tendencias fx maxilo/data_raw")   # EGRE_DATOS_ABIERTOS_{anio}.csv
GRD_DIR  = Path(r"C:/Users/gusta/Downloads/economic burden del trauma")              # GRD_PUBLICO_{anio}.txt
INE_FILE = GRD_DIR / "estimaciones-y-proyecciones-2002-2035-comuna-y-área-urbana-y-rural.xlsx"
IPC_FILE = GRD_DIR / "Indicador.xls"                                                 # BCCh, variación % mensual
PROD_FILE = GRD_DIR / "parametros_productividad_2019_2024.csv"                       # opcional (costos indirectos)
# Parámetros anuales (IPC, productividad): se usan los mismos archivos de los artículos. Los años de egresos sin
# parámetro (p. ej. 2025 con productividad hasta 2024) toman el último año disponible; actualízalos cuando salgan.
# Nacidos vivos: el repositorio ya trae DEIS 2001-2023 e INE provisional 2024-2025. Agrega aquí los CSV nuevos
# de cada año (anio, region, nacidos_vivos; sin región = cifra nacional).
NACIMIENTOS_EXTRA = []
DEIS_PATRON = "EGRE_DATOS_ABIERTOS_{anio}.csv"
GRD_PATRON  = "GRD_PUBLICO_{anio}.txt"
ANIOS_DEIS = [2001, None]   # None = hasta el último archivo que exista
ANIOS_GRD  = [2019, None]
RSCRIPT = None              # ruta a Rscript.exe si no está en el PATH, p. ej. r"C:/Program Files/R/R-4.4.1/bin/Rscript.exe"

TRABAJO = REPO_DIR.parent / "observatorio_corrida_anual"   # copia de trabajo (conserva la caché entre años)
SALIDA  = REPO_DIR.parent
print("Repositorio:", REPO_DIR, (REPO_DIR / "pipeline").exists())
for nombre, p in [("DEIS", DEIS_DIR), ("GRD", GRD_DIR), ("INE", INE_FILE), ("IPC", IPC_FILE), ("Productividad", PROD_FILE)]:
    print(f"  {nombre:<13} {'✓' if p.exists() else '✗ no encontrado'}  {p}")
print("  DEIS:", len(list(DEIS_DIR.glob("EGRE_DATOS_ABIERTOS_*.csv"))), "archivos, GRD:",
      len(list(GRD_DIR.glob("GRD_PUBLICO_*.txt"))), "archivos")

Repositorio: c:\Users\gusta\Downloads\observatoriocmf\observatoriocmf True
  DEIS          ✓  C:\Users\gusta\Downloads\Estudio tendencias fx maxilo\data_raw
  GRD           ✓  C:\Users\gusta\Downloads\economic burden del trauma
  INE           ✓  C:\Users\gusta\Downloads\economic burden del trauma\estimaciones-y-proyecciones-2002-2035-comuna-y-área-urbana-y-rural.xlsx
  IPC           ✓  C:\Users\gusta\Downloads\economic burden del trauma\Indicador.xls
  Productividad ✓  C:\Users\gusta\Downloads\economic burden del trauma\parametros_productividad_2019_2024.csv
  DEIS: 25 archivos, GRD: 7 archivos


## 0. *(Una vez)* Guardar los parámetros de los artículos en el repositorio

Copia a `data-small/` del repositorio los archivos de población INE, IPC y productividad que usaste en los
artículos y los registra en `config/sources.yml`. Así GitHub puede recalcular sin tu computador, y desde ahí
los actualizas en el panel. Si ya están registrados, no hace nada.

In [2]:
import re
def registrar_parametro(fuente, archivo, etiqueta):
    if not Path(archivo).exists():
        print(f"  {fuente:<14} ✗ no encontrado: {archivo}"); return
    yml_path = REPO_DIR / "config" / "sources.yml"
    yml = yml_path.read_text(encoding="utf-8")
    m = re.search(rf"(?ms)^  {fuente}:\n.*?(?=^  \S|\Z)", yml)
    if not m:
        print(f"  {fuente:<14} ✗ fuente no definida en sources.yml"); return
    bloque = m.group(0)
    if not re.search(r"(?m)^    versiones: \[\][ \t]*$", bloque):
        print(f"  {fuente:<14} ya registrado (se actualiza desde el panel)"); return
    destino = REPO_DIR / "data-small" / fuente / f"{etiqueta}{Path(archivo).suffix.lower()}"
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(archivo, destino)
    rel = destino.relative_to(REPO_DIR).as_posix()
    nuevo = re.sub(r"(?m)^    versiones: \[\][ \t]*$",
                   f"    versiones:\n      - version: \"{etiqueta}\"\n        archivo: {rel}\n"
                   f"        notas: Archivo usado en los artículos (registrado desde el notebook de actualización).",
                   bloque, count=1)
    yml_path.write_text(yml.replace(bloque, nuevo), encoding="utf-8")
    print(f"  {fuente:<14} ✓ {rel}")

registrar_parametro("poblacion_ine", INE_FILE, "INE-2017-base")
registrar_parametro("ipc", IPC_FILE, "BCCh-" + datetime.date.fromtimestamp(IPC_FILE.stat().st_mtime).strftime("%Y%m") if IPC_FILE.exists() else "BCCh")
registrar_parametro("productividad", PROD_FILE, PROD_FILE.stem.replace("parametros_productividad_", "ESI-"))

  poblacion_ine  ✓ data-small/poblacion_ine/INE-2017-base.xlsx
  ipc            ✓ data-small/ipc/BCCh-202609.xls
  productividad  ✓ data-small/productividad/ESI-2019_2024.csv


## 1. Procesar

In [3]:
def find_rscript():
    if RSCRIPT and Path(RSCRIPT).exists(): return str(RSCRIPT)
    w = shutil.which("Rscript")
    if w: return w
    for base in [Path(r"C:/Program Files/R"), Path.home() / "AppData/Local/Programs/R"]:
        c = sorted(base.glob("R-*/bin/Rscript.exe")) if base.exists() else []
        if c: return str(c[-1])
    raise FileNotFoundError("No encontré Rscript: define RSCRIPT")

# Copia de trabajo: código y configuración nuevos del repositorio; caché de años anteriores se conserva.
TRABAJO.mkdir(exist_ok=True)
for sub in ["R", "pipeline", "config", "dictionaries", "data-small", "inst"]:
    if (REPO_DIR / sub).exists():
        shutil.copytree(REPO_DIR / sub, TRABAJO / sub, dirs_exist_ok=True)
for f in ["DESCRIPTION", "NAMESPACE", "CITATION.cff", "NEWS.md"]:
    if (REPO_DIR / f).exists(): shutil.copy(REPO_DIR / f, TRABAJO / f)

P = lambda p: str(p).replace("\\", "/")
loc = {"deis_dir": P(DEIS_DIR), "deis_patron": DEIS_PATRON, "grd_dir": P(GRD_DIR), "grd_patron": GRD_PATRON,
       "ine": P(INE_FILE) if INE_FILE.exists() else None, "ipc": P(IPC_FILE) if IPC_FILE.exists() else None,
       "productividad": P(PROD_FILE) if PROD_FILE.exists() else None,
       "nacimientos_extra": [P(x) for x in NACIMIENTOS_EXTRA],
       "anios_deis": ANIOS_DEIS, "anios_grd": ANIOS_GRD}
(TRABAJO / "config_local.json").write_text(json.dumps(loc, ensure_ascii=False, indent=2), encoding="utf-8")

# Windows no reconoce LANG=C.UTF-8 (R caería en el locale "C" y leería mal los acentos): se deja el de R,
# que desde R 4.2 ya es UTF-8. En Linux/macOS se fuerza UTF-8.
env = dict(os.environ)
if os.name == "nt":
    for k in ("LANG", "LC_ALL", "LC_CTYPE"): env.pop(k, None)
else:
    env.update(LANG="C.UTF-8", LC_ALL="C.UTF-8")
ver = subprocess.run([find_rscript(), "-e", "cat(R.version.string, l10n_info()[['UTF-8']])"], capture_output=True,
                     text=True, env=env)
print("R:", ver.stdout.strip() or ver.stderr.strip())
inicio = datetime.datetime.now()
print("Procesando… (la primera vez puede tardar bastante; los años siguientes salen de la caché)")
p = subprocess.run([find_rscript(), "pipeline/actualizar_local.R", "config_local.json"], cwd=TRABAJO,
                   capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
log = p.stdout + "\n" + p.stderr
(TRABAJO / "log_actualizacion.txt").write_text(log, encoding="utf-8")
print("\n".join(l for l in log.splitlines() if "Procesando" not in l)[-4000:])
print(f"Duración: {datetime.datetime.now() - inicio}")
assert p.returncode == 0, "El pipeline terminó con error: revisa log_actualizacion.txt (arriba van las últimas líneas)"
assert (TRABAJO / "inst/extdata/controles.json").exists(), "No se generaron los controles: revisa log_actualizacion.txt"

R: R version 4.5.1 (2025-06-13 ucrt) TRUE
Procesando… (la primera vez puede tardar bastante; los años siguientes salen de la caché)
[14:12:30] Observatorio CMF Chile: pipeline
[14:43:12] Casos clasificados guardados en data-casos/ (32 archivos)
[14:43:14] Aviso: IPC con menos de 12 meses en 2026
[14:43:17] Aviso: egresos_deis 2023: 8.75% sin región resuelta
[14:43:17] Aviso: egresos_deis 2025: 7.97% sin región resuelta
[14:43:17] Aviso: Años sin población INE (sin tasas): 2001
[14:43:18] Exportado a inst/extdata (csv): cubo, universo, totales, estada, severidad, conteos_comuna, poblacion, poblacion_edad0, ipc, oferta_servicios, dic_diagnosticos, dic_procedimientos, dic_causas_externas, dic_regiones, dic_servicios_salud, dic_comunas, poblacion_estandar, mapa_regiones, mapa_servicios_salud
[14:43:19] Controles contra publicaciones: coincide 25, difiere 0, sin datos 0
[14:43:19] Listo.

Registrados: 25 años DEIS, 7 años GRD
Aviso:
package 'data.table' was built under R version 4.5.2 
Dura

## 2. Revisar: controles contra las publicaciones y validación

In [4]:
import pandas as pd
ext = TRABAJO / "inst" / "extdata"
m = json.loads((ext / "manifest.json").read_text(encoding="utf-8"))
print("Años procesados:", m.get("anios"), ", errores:", m.get("n_errores"), ", advertencias:", m.get("n_advertencias"))
ct = pd.DataFrame(json.loads((ext / "controles.json").read_text(encoding="utf-8")))
pd.set_option("display.max_colwidth", 90)
display(ct[["referencia", "anio_desde", "anio_hasta", "valor", "obtenido", "estado"]])
# "difiere" en años ya publicados suele indicar que el DEIS revisó un archivo antiguo o que el INE cambió
# las proyecciones (denominadores). Revisa antes de publicar; los años nuevos no tienen control.
# Filas leídas por base y año (deben coincidir con el diagnóstico; p. ej. GRD 2022 = 932.840).
qc = json.loads((TRABAJO / "data-casos" / "qc.json").read_text(encoding="utf-8"))
qc = list(qc.values()) if isinstance(qc, dict) else qc
filas = pd.DataFrame([{"fuente": q.get("fuente"), "anio": q.get("version"), "filas": q.get("n_leidos"),
                       "hospitalizacion": q.get("n_hospitalizacion"), "sin_region_%": q.get("pct_sin_region")} for q in qc])
display(filas.pivot_table(index="anio", columns="fuente", values="filas", aggfunc="first"))
v = json.loads((ext / "validacion.json").read_text(encoding="utf-8"))
for a in v.get("advertencias", [])[:30]: print("•", a)

Años procesados: {'egresos_deis': [2001, 2025], 'grd': [2019, 2025]} , errores: 0 , advertencias: 3


,referencia,anio_desde,anio_hasta,valor,obtenido,estado
0,"Infecciones cabeza y cuello 2002-2025, Tabla 1: total",2002,2025,153408.000,153408.000000,coincide
1,"Infecciones, Tabla 1: periamigdalina",2002,2025,74606.000,74606.000000,coincide
2,"Infecciones, Tabla 1: piel y tejidos blandos",2002,2025,41607.000,41607.000000,coincide
3,"Infecciones, Tabla 1: oral y maxilares",2002,2025,20606.000,20606.000000,coincide
4,"Infecciones, Tabla 1: orbitaria",2002,2025,7356.000,7356.000000,coincide
5,"Infecciones, Tabla 1: glándulas salivales",2002,2025,5648.000,5648.000000,coincide
6,"Infecciones, Tabla 1: faríngea/cervical profunda",2002,2025,3585.000,3585.000000,coincide
7,"Infecciones, Resultados: tasa nacional 2002",2002,2002,47.300,47.324379,coincide
8,"Infecciones, Resultados: tasa nacional 2019",2019,2019,32.600,32.584548,coincide
9,"Infecciones, Resultados: tasa nacional 2021",2021,2021,16.800,16.790014,coincide


fuente,egresos_deis,grd
anio,,
2001,1566188.0,NaN
2002,1599076.0,NaN
2003,1599280.0,NaN
2004,1627748.0,NaN
2005,1627743.0,NaN
2006,1637920.0,NaN
2007,1632888.0,NaN
2008,1608540.0,NaN
2009,1682054.0,NaN


• egresos_deis 2023: 8.75% sin región resuelta
• egresos_deis 2025: 7.97% sin región resuelta
• Años sin población INE (sin tasas): 2001


## 3. Empaquetar para subir al panel

In [5]:
fecha = datetime.date.today().strftime("%Y%m%d")
destino = SALIDA / f"datos_procesados_observatorio_{fecha}"
import zipfile
zip_path = str(destino) + ".zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in ["inst/extdata", "data-casos"]:
        for f in sorted((TRABAJO / carpeta).rglob("*")):
            if f.is_file(): z.write(f, f.relative_to(TRABAJO).as_posix())
print("Listo:", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
print("Súbelo en el panel → Publicar → Actualización anual; luego Desplegar y versionar (minor).")
n_casos = len(list((TRABAJO / "data-casos").glob("casos_*")))
print(f"Incluye {n_casos} archivos de casos clasificados (por fuente y año) y las tablas publicadas.")

Listo: c:\Users\gusta\Downloads\observatoriocmf\datos_procesados_observatorio_20261110.zip (9.9 MB)
Súbelo en el panel → Publicar → Actualización anual; luego Desplegar y versionar (minor).
Incluye 32 archivos de casos clasificados (por fuente y año) y las tablas publicadas.


## 4. Ver la app en tu computador (antes de subir)

Abre la app con los datos recién procesados en http://127.0.0.1:8765 . Sigue corriendo mientras el kernel esté
abierto; para cerrarla ejecuta `app.terminate()`.

In [6]:
import webbrowser, time
app = subprocess.Popen([find_rscript(), "-e", "shiny::runApp('inst/app', port = 8765, launch.browser = FALSE)"],
                       cwd=TRABAJO, env=env)
time.sleep(8)
webbrowser.open("http://127.0.0.1:8765")
print("App en http://127.0.0.1:8765 ,  para cerrarla: app.terminate()")

App en http://127.0.0.1:8765 ,  para cerrarla: app.terminate()


## 5. Repositorio y carpeta para GitHub

Copia los casos clasificados y las tablas al repositorio y arma `observatoriocmf_github` (junto al repositorio), una
carpeta limpia para arrastrar en github.com o abrir con GitHub Desktop: sin bases DEIS/GRD, caché ni rutas de tu
computador. `CON_CASOS = True` agrega `data-casos/` (necesario para *Recalcular* en GitHub).


In [7]:
import sys
sys.path.insert(0, str(REPO_DIR / "pipeline"))
import importlib, cmf_local as cmf; importlib.reload(cmf)
cmf.publicar_en_repo(TRABAJO, REPO_DIR)
CON_CASOS = False
cmf.carpeta_github(REPO_DIR, con_casos=CON_CASOS)

Repositorio listo en c:\Users\gusta\Downloads\observatoriocmf\observatoriocmf (17 MB). No contiene las bases DEIS/GRD ni rutas de tu computador.
